### SELECT & WHERE

In [0]:
%sql
select customer_id,customer_name,city,state,customer_segment from databricks_prep.data.customers

### Find all Premium customers

In [0]:
%sql
select * from databricks_prep.data.customers where customer_segment = 'Premium'

### Find customers from Bangalore

In [0]:
%sql
select * from databricks_prep.data.customers where city = 'Bangalore'

### Find Electronics orders

In [0]:
%sql
select * from databricks_prep.data.orders where category = 'Electronics'

### Find orders with unit price greater than 1000

In [0]:
%sql
select * from databricks_prep.data.orders where unit_price > 1000 order by unit_price desc
    

### Find completed orders

In [0]:
%sql
select * from databricks_prep.data.orders where status = "Completed"

### Find completed Electronics orders above ₹1,000

In [0]:
%sql
select * from databricks_prep.data.orders where status = "Completed" and category ="Electronics" and unit_price > 1000

### Calculate order value

In [0]:
%sql
select *, quantity * unit_price as total_price from databricks_prep.data.orders

### Calculate net order value

In [0]:
%sql
select order_id, quantity*unit_price as gross_amount,gross_amount-discount as net_amount from databricks_prep.data.orders

### Categorize orders

In [0]:
%sql
select order_id,quantity*unit_price as gross_amount,gross_amount-discount as net_amount,
case 
    when net_amount>=2000 then 'High Value' 
    when net_amount>=1000 then 'Medium Value' 
    else 'Low Value' 
    end as category 
from databricks_prep.data.orders

### Count orders by category

In [0]:
%sql
select category,count(*) from databricks_prep.data.orders group by category

### Find total sales by category

In [0]:
%sql
select category, sum(quantity*unit_price) as total_sales from databricks_prep.data.orders group by category
    

### Find average order value by category

In [0]:
%sql
select category, avg(quantity*unit_price) as avg_sales from databricks_prep.data.orders group by category
    

### Find minimum and maximum order value by category

In [0]:
%sql
select category, min(quantity*unit_price) as min_sales,max(quantity*unit_price) as max_sales from databricks_prep.data.orders group by category
    

### Find total quantity sold by category

In [0]:
%sql
select category,sum(quantity) as total_quantity from databricks_prep.data.orders group by category

### Find customers with more than one order

In [0]:
%sql
select c.customer_id, count(*) as order_count from databricks_prep.data.customers c join databricks_prep.data.orders o on c.customer_id = o.customer_id group by c.customer_id having count(*)> 1

### Find customers whose total revenue is greater than 2500

In [0]:
%sql
select c.customer_id, sum(quantity*unit_price) as total_revenue from databricks_prep.data.customers c join databricks_prep.data.orders o on c.customer_id = o.customer_id group by c.customer_id having total_revenue>2500

# Join

### Join customers and orders

In [0]:
%sql
select c.customer_name,c.city,c.customer_segment,o.order_id,o.category,o.quantity,o.unit_price from databricks_prep.data.customers c join databricks_prep.data.orders o on c.customer_id = o.customer_id

### Find total revenue per customer

In [0]:
%sql
select c.customer_id, c.customer_name, sum(quantity*unit_price) as total_revenue from databricks_prep.data.customers c join databricks_prep.data.orders o on c.customer_id = o.customer_id group by c.customer_id, c.customer_name

### Find revenue by city

In [0]:
%sql
select c.city,sum(o.quantity) as total_orders, sum(quantity*unit_price) as total_revenue from databricks_prep.data.customers c join databricks_prep.data.orders o on c.customer_id = o.customer_id group by c.city

### Find revenue by customer segment

In [0]:
%sql
select c.customer_segment,count(c.customer_id) as total_customers,sum(o.quantity) as total_orders, sum(quantity*unit_price) as total_revenue from databricks_prep.data.customers c join databricks_prep.data.orders o on c.customer_id = o.customer_id group by c.customer_segment;


### Find Premium customers who placed orders

In [0]:
%sql
select c.customer_name,c.customer_segment,o.order_id,o.category from databricks_prep.data.customers c join databricks_prep.data.orders o on c.customer_id = o.customer_id where c.customer_segment = 'Premium'

### Find customers who never placed an order

In [0]:
%sql
select * from databricks_prep.data.customers left join databricks_prep.data.orders on databricks_prep.data.customers.customer_id = databricks_prep.data.orders.customer_id where databricks_prep.data.orders.order_id is null;

### Find customers with completed orders only

In [0]:
%sql
select o.customer_id,c.customer_name,count(*) AS total_completed_orders,sum(o.quantity*o.unit_price) AS total_revenue from databricks_prep.data.customers c join databricks_prep.data.orders o on c.customer_id = o.customer_id where o.status = "Completed" group by o.customer_id,c.customer_name

# Window Functions

### Rank employees by salary

In [0]:
%sql
select employee_name,department,salary,RANK() OVER (ORDER BY salary DESC) AS salary_rank from databricks_prep.data.employees

### Rank employees within each department

In [0]:
%sql
select employee_name,department,salary,RANK() OVER (PARTITION BY department order by salary DESC) AS deparment_rank from databricks_prep.data.employees


### Find highest-paid employee in each department

In [0]:
%sql
select department,employee_name,salary from (
  select department,employee_name,salary,row_number() over (PARTITION BY department order by salary desc) AS row_number from databricks_prep.data.employees
)
where row_number = 1;


### Find top 2 employees in each department

In [0]:
%sql
select * from (
  select *,row_number() over (PARTITION BY department order by salary desc) AS row_number from databricks_prep.data.employees
)
where row_number <= 2;

### Find second-highest salary in each department

In [0]:
%sql
select * from(select *,dense_rank() over (partition by department order by salary desc)  rnk from databricks_prep.data.employees )where rnk = 2

### Find the second-highest salary overall

In [0]:
%sql
select * from(select *,row_number() over (order by salary desc) rnk from databricks_prep.data.employees) where rnk = 2

### Using Limit 2

In [0]:
%sql
select * from databricks_prep.data.employees order by salary desc limit 1 offset 2

### Sub query

In [0]:
%sql
select max(salary) from databricks_prep.data.employees where salary <(select max(salary) from databricks_prep.data.employees)

### Rank customers by revenue

In [0]:
%sql
select customer_name, total_revenue, 
       rank() over(order by total_revenue desc) as revenue_rank
from (
  select c.customer_name, 
         sum((o.quantity*o.unit_price)-o.discount) as total_revenue
  from databricks_prep.data.orders o
  join databricks_prep.data.customers c on o.customer_id = c.customer_id
  group by c.customer_name
)

### Find top 3 customers by revenue

In [0]:
%sql
select customer_name, total_revenue, revenue_rank
from (
  select customer_name, total_revenue, 
         dense_rank() over(order by total_revenue desc) as revenue_rank
  from (
    select c.customer_name, 
           sum((o.quantity*o.unit_price)-o.discount) as total_revenue
    from databricks_prep.data.orders o
    join databricks_prep.data.customers c on o.customer_id = c.customer_id
    group by c.customer_name
  )
)
where revenue_rank <= 3

### Find highest-value order for every customer

In [0]:
%sql
with ranked_orders AS (
    SELECT 
        o.customer_id,
        c.customer_name,
        o.order_id,
        sum(quantity*unit_price) as order_value,
        row_number() over(partition by o.customer_id order by sum(quantity*unit_price) desc) as order_rank
    FROM 
        databricks_prep.data.orders o
    JOIN 
        databricks_prep.data.customers c
    ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name, o.order_id
)
select customer_id, customer_name, order_id, order_value
from ranked_orders
where order_rank =1;

### Find second-highest order for every customer

In [0]:
%sql
with ranked_orders AS (
    SELECT 
        o.customer_id,
        c.customer_name,
        o.order_id,
        sum(quantity*unit_price) as order_value,
        dense_rank() over(partition by o.customer_id order by sum(quantity*unit_price) desc) as order_rank
    FROM 
        databricks_prep.data.orders o
    JOIN 
        databricks_prep.data.customers c
    ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name, o.order_id
)
select customer_id, customer_name, order_id, order_value
from ranked_orders
where order_rank =2;

### Rank orders within each category

In [0]:
%sql
with orders_data as (
    select category,order_id,sum(quantity*unit_price) as order_value,rank() over(partition by category order by sum(quantity*unit_price) desc) as rank
    from databricks_prep.data.orders
    group by category,order_id
)
select * from orders_data

###Find top 2 orders from each category

In [0]:
%sql
with cte as(
    select category,order_id,sum(quantity*unit_price) AS order_value,rank() over(partition by category order by sum(quantity*unit_price) desc) as rnk
    from databricks_prep.data.orders
    group by category,order_id
)
select * from cte
where rnk<=2

### Find previous order value

In [0]:
%sql
SELECT customer_id, order_date, total_value,
    LAG(total_value) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS previous_order_value
FROM (
    SELECT customer_id, order_date, SUM(quantity*unit_price) AS total_value
    FROM databricks_prep.data.orders
    GROUP BY customer_id, order_date
)

### Find next order value

In [0]:
%sql
SELECT customer_id, order_date, total_value,
    LEAD(total_value) OVER (
        PARTITION BY customer_id
        ORDER BY order_date
    ) AS previous_order_value
FROM (
    SELECT customer_id, order_date, SUM(quantity*unit_price) AS total_value
    FROM databricks_prep.data.orders
    GROUP BY customer_id, order_date
)

### Find customers whose current order is greater than their previous order

In [0]:
%sql
WITH order_comparison AS (
    SELECT
        customer_id,
        order_date,
        order_value,
        LAG(order_value) OVER (
            PARTITION BY customer_id
            ORDER BY order_date
        ) AS previous_order_value
    FROM (
        SELECT customer_id, order_date, SUM(quantity*unit_price) AS order_value
        FROM databricks_prep.data.orders
        GROUP BY customer_id, order_date
    )
)

SELECT
    customer_id,
    order_date,
    order_value,
    previous_order_value,
    CASE
        WHEN previous_order_value IS NOT NULL AND order_value > previous_order_value
        THEN 'Yes'
        ELSE 'No'
    END AS order_increased
FROM order_comparison

### Calculate difference between current and previous order